# 🎓 Student Performance Predictor
A machine learning model that flags **at-risk students early** using attendance and test-score trends.

**Tech stack:** Python · Pandas · Scikit-learn (+ Matplotlib for charts)

**How to use:** Run the cells from top to bottom (`Shift + Enter`). The notebook is self-contained: if `data/students.csv` is not found, it generates sample data automatically.

**Steps**
1. Load data
2. Build trend features
3. Train and compare models
4. Evaluate
5. Predict for new students
6. Look at one student in detail

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve

%matplotlib inline
pd.set_option("display.max_columns", 30)
os.makedirs("data", exist_ok=True)
os.makedirs("outputs", exist_ok=True)
print("Libraries loaded")

## 1. Load the data
If `data/students.csv` exists it is used. Otherwise sample data is generated and saved.

In [ ]:
WEEKS, TESTS = 8, 5

def make_students(n, seed, with_label=True):
    rng = np.random.default_rng(seed)
    ability = rng.normal(0, 1, n)  # hidden overall strength of each student

    att_base = np.clip(80 + 8 * ability + rng.normal(0, 5, n), 40, 100)
    att_trend = rng.normal(0, 1.2, n) + 0.5 * ability
    w = np.arange(WEEKS)
    att = np.clip(att_base[:, None] + att_trend[:, None] * (w - WEEKS / 2)
                  + rng.normal(0, 3, (n, WEEKS)), 0, 100)

    sc_base = 65 + 12 * ability + rng.normal(0, 6, n)
    sc_trend = rng.normal(0, 2, n) + 1.0 * ability
    t = np.arange(TESTS)
    sc = np.clip(sc_base[:, None] + sc_trend[:, None] * (t - TESTS / 2)
                 + rng.normal(0, 4, (n, TESTS)), 0, 100)

    assignment = np.clip(75 + 10 * ability + rng.normal(0, 8, n), 0, 100)

    df = pd.DataFrame({"student_id": [f"S{seed}{i:04d}" for i in range(n)]})
    for i in range(WEEKS):
        df[f"attendance_w{i+1}"] = att[:, i].round(1)
    for i in range(TESTS):
        df[f"test{i+1}"] = sc[:, i].round(1)
    df["assignment_completion"] = assignment.round(1)
    if with_label:
        final = 62 + 14 * ability + 2.0 * att_trend + 1.5 * sc_trend + rng.normal(0, 6, n)
        df["at_risk"] = (final < 50).astype(int)   # 1 = at-risk
    return df

if os.path.exists("data/students.csv"):
    df = pd.read_csv("data/students.csv")
    print("Loaded data/students.csv")
else:
    df = make_students(1000, seed=42)
    df.to_csv("data/students.csv", index=False)
    print("Generated sample data and saved to data/students.csv")

if not os.path.exists("data/new_students.csv"):
    make_students(200, seed=7, with_label=False).to_csv("data/new_students.csv", index=False)

print(df.shape)
df.head()

In [ ]:
print("Missing values:", int(df.isna().sum().sum()))
print("At-risk students: {:.1f}%".format(df["at_risk"].mean() * 100))
df.describe().T.round(1)

In [ ]:
counts = df["at_risk"].value_counts().sort_index()
fig, ax = plt.subplots(figsize=(4.5, 3.2))
ax.bar(["On-track", "At-risk"], counts.values, color=["#2e9e5b", "#d64545"])
for i, v in enumerate(counts.values):
    ax.text(i, v, str(v), ha="center", va="bottom")
ax.set_title("Class balance")
ax.set_ylabel("Students")
plt.show()

## 2. Build trend features
A single average hides important information. A student with 80% attendance that is **falling every week** is riskier than one with 80% that is steady. So besides averages we compute **slopes** (trends) using a straight-line fit.

In [ ]:
def _slope(matrix):
    # slope of each row: positive = improving, negative = declining
    x = np.arange(matrix.shape[1])
    return np.polyfit(x, matrix.T, 1)[0]

def build_features(df):
    att_cols = sorted([c for c in df.columns if c.startswith("attendance_w")],
                      key=lambda c: int(c.split("_w")[1]))
    test_cols = sorted([c for c in df.columns if c.startswith("test")],
                       key=lambda c: int(c.replace("test", "")))
    att = df[att_cols].to_numpy(dtype=float)
    sc = df[test_cols].to_numpy(dtype=float)

    f = pd.DataFrame(index=df.index)
    f["att_mean"] = att.mean(axis=1)
    f["att_min"] = att.min(axis=1)
    f["att_slope"] = _slope(att)
    f["att_recent_drop"] = att[:, -3:].mean(axis=1) - att[:, :3].mean(axis=1)
    f["score_mean"] = sc.mean(axis=1)
    f["score_last"] = sc[:, -1]
    f["score_std"] = sc.std(axis=1)
    f["score_slope"] = _slope(sc)
    f["score_change"] = sc[:, -1] - sc[:, 0]
    if "assignment_completion" in df.columns:
        f["assignment_completion"] = df["assignment_completion"]
    return f

X = build_features(df)
y = df["at_risk"]
X.head().round(2)

In [ ]:
# How do at-risk and on-track students differ?
X.groupby(y.map({0: "On-track", 1: "At-risk"})).mean().T.round(2)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
for label, color, name in [(0, "#2e9e5b", "On-track"), (1, "#d64545", "At-risk")]:
    axes[0].hist(X.loc[y == label, "att_slope"], bins=25, alpha=0.6, color=color, label=name)
    axes[1].hist(X.loc[y == label, "score_slope"], bins=25, alpha=0.6, color=color, label=name)
axes[0].set_title("Attendance trend (slope per week)")
axes[1].set_title("Test-score trend (slope per test)")
for a in axes:
    a.axvline(0, color="black", linewidth=0.8)
    a.legend()
plt.tight_layout()
plt.show()

## 3. Train and compare models
We split the data 80/20 and compare two models using 5-fold cross-validation (ROC-AUC).

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

models = {
    "LogisticRegression": make_pipeline(
        StandardScaler(), LogisticRegression(max_iter=1000, class_weight="balanced")),
    "RandomForest": RandomForestClassifier(
        n_estimators=300, max_depth=8, class_weight="balanced", random_state=42),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = {}
for name, model in models.items():
    scores[name] = cross_val_score(model, X_train, y_train, cv=cv, scoring="roc_auc").mean()
    print(f"{name:20s} CV ROC-AUC: {scores[name]:.3f}")

best_name = max(scores, key=scores.get)
best = models[best_name].fit(X_train, y_train)
print("\nBest model:", best_name)

## 4. Evaluate
We want to **catch at-risk students**, so we use a flag threshold of **0.4** instead of 0.5. This raises recall at the cost of a few extra false alarms.

In [ ]:
THRESHOLD = 0.4
proba = best.predict_proba(X_test)[:, 1]
pred = (proba >= THRESHOLD).astype(int)

print(f"Test ROC-AUC: {roc_auc_score(y_test, proba):.3f}\n")
print(classification_report(y_test, pred, target_names=["On-track", "At-risk"]))

In [ ]:
cm = confusion_matrix(y_test, pred)
fpr, tpr, _ = roc_curve(y_test, proba)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))

axes[0].imshow(cm, cmap="Blues")
axes[0].set_xticks([0, 1]); axes[0].set_yticks([0, 1])
axes[0].set_xticklabels(["On-track", "At-risk"]); axes[0].set_yticklabels(["On-track", "At-risk"])
axes[0].set_xlabel("Predicted"); axes[0].set_ylabel("Actual")
axes[0].set_title("Confusion matrix")
for i in range(2):
    for j in range(2):
        axes[0].text(j, i, cm[i, j], ha="center", va="center",
                     color="white" if cm[i, j] > cm.max() / 2 else "black", fontsize=14)

axes[1].plot(fpr, tpr, color="#4a6fa5", label=f"AUC = {roc_auc_score(y_test, proba):.3f}")
axes[1].plot([0, 1], [0, 1], "--", color="gray")
axes[1].set_xlabel("False positive rate"); axes[1].set_ylabel("True positive rate")
axes[1].set_title("ROC curve"); axes[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
# Which features drive the risk?
final_model = best[-1] if hasattr(best, "steps") else best
if hasattr(final_model, "feature_importances_"):
    imp = pd.Series(final_model.feature_importances_, index=X.columns)
else:
    imp = pd.Series(np.abs(final_model.coef_[0]), index=X.columns)
imp = imp.sort_values()

fig, ax = plt.subplots(figsize=(6, 4))
ax.barh(imp.index, imp.values, color="#4a6fa5")
ax.set_title("Feature importance")
plt.tight_layout()
plt.show()

## 5. Predict for new students
This scores the students in `data/new_students.csv` (no label) and ranks them by risk. You can replace this file with your own data.

In [ ]:
new = pd.read_csv("data/new_students.csv")
Xn = build_features(new)[X.columns]

new["risk_probability"] = best.predict_proba(Xn)[:, 1].round(3)
new["risk_level"] = pd.cut(new["risk_probability"], [-0.01, 0.3, 0.6, 1.0],
                           labels=["Low", "Medium", "High"])
new["flag_at_risk"] = (new["risk_probability"] >= THRESHOLD).astype(int)

result = (new[["student_id", "risk_probability", "risk_level", "flag_at_risk"]]
          .sort_values("risk_probability", ascending=False))
result.to_csv("outputs/at_risk_students.csv", index=False)

print(result["risk_level"].value_counts().to_string())
print("\nSaved to outputs/at_risk_students.csv")
result.head(10)

In [ ]:
colors = {"Low": "#2e9e5b", "Medium": "#f0a30a", "High": "#d64545"}
counts = result["risk_level"].value_counts().reindex(["Low", "Medium", "High"], fill_value=0)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].bar(counts.index, counts.values, color=[colors[k] for k in counts.index])
for i, v in enumerate(counts.values):
    axes[0].text(i, v, str(v), ha="center", va="bottom")
axes[0].set_title("Risk level distribution")

axes[1].hist(new["risk_probability"], bins=20, color="#4a6fa5", edgecolor="white")
axes[1].axvline(THRESHOLD, color="red", linestyle="--", label=f"Threshold {THRESHOLD}")
axes[1].set_title("Risk probability distribution")
axes[1].legend()
plt.tight_layout()
plt.show()

## 6. Look at one student in detail
Change `student_id` to any ID from the table above.

In [ ]:
student_id = result.iloc[0]["student_id"]   # highest-risk student; change to any ID

row = new[new["student_id"] == student_id].iloc[0]
att_cols = [c for c in new.columns if c.startswith("attendance_w")]
test_cols = [c for c in new.columns if c.startswith("test")]

print(f"Student {student_id}: risk = {row['risk_probability']*100:.1f}%  ({row['risk_level']})")

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].plot(range(1, len(att_cols)+1), row[att_cols].astype(float), marker="o",
             color=colors[str(row["risk_level"])], label=student_id)
axes[0].plot(range(1, len(att_cols)+1), new[att_cols].mean(), "--", color="gray", label="Class average")
axes[0].set_title("Weekly attendance %"); axes[0].set_xlabel("Week"); axes[0].legend()

axes[1].plot(range(1, len(test_cols)+1), row[test_cols].astype(float), marker="o",
             color=colors[str(row["risk_level"])], label=student_id)
axes[1].plot(range(1, len(test_cols)+1), new[test_cols].mean(), "--", color="gray", label="Class average")
axes[1].set_title("Test scores"); axes[1].set_xlabel("Test"); axes[1].legend()
plt.tight_layout()
plt.show()

## Summary
- Trend features (slopes, recent drop, score change) capture whether a student is improving or declining.
- The model is tuned with a lower threshold (0.4) so that at-risk students are not missed.
- Results are saved to `outputs/at_risk_students.csv` for lecturers or mentors to follow up.

**Next ideas:** add more features (assignment trends, subject-wise scores), try gradient boosting, and add intervention suggestions per student.